# Estudo de Estabilidade Global de Estruturas Pré-Moldadas
**Análise Interativa de 1ª e 2ª Ordem (Método de Hogeslag e Gama-z)**

Este notebook implementa a análise de estabilidade global solicitada, estruturando as comparações de carregamento e rigidez de fundações[cite: 1, 3]. Como premissa, as vigas são admitidas como axialmente indeformáveis e perfeitamente articuladas nos pilares, permitindo a condensação do pórtico em um pilar equivalente[cite: 2].

## 1. Avaliação de Deslocamentos e Coeficiente Gama-z ($\gamma_z$)
O bloco abaixo resolve a **Estrutura E2 (2 pavimentos)** para a **Combinação C2** (Principal Sobrecarga, Secundário Vento)[cite: 4]. A inércia da seção é reduzida para $EI/3$ (simulando a não-linearidade física exigida nos métodos aproximados)[cite: 5]. O simulador permite alternar a fundação entre um engaste perfeito (F1) e uma mola rotacional (F2)[cite: 3].

In [1]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display

def analisar_estrutura_e2(tipo_fundacao):
    # 1. Parâmetros Base e Redução de Rigidez
    E_c = 30e6          
    I_eq = 4 * ((0.40**4)/12)
    H = 6.0             
    EI = (E_c * I_eq) / 3 
    
    # 2. Cargas (Combinação C2)
    F_vd_pav = 1.4 * 447 + 1.4 * 189
    W_d = 0.84 * np.array([26.67, 53.33])
    
    # 3. Matriz de Rigidez Local
    def rigidez_elemento(EI, L):
        return (EI / L**3) * np.array([
            [12,   6*L,     -12,  6*L],
            [6*L,  4*L**2,  -6*L, 2*L**2],
            [-12, -6*L,      12, -6*L],
            [6*L,  2*L**2,  -6*L, 4*L**2]
        ])

    K_global = np.zeros((6, 6))
    K_global[0:4, 0:4] += rigidez_elemento(EI, H)
    K_global[2:6, 2:6] += rigidez_elemento(EI, H)
    
    # 4. Condições de Contorno e Interface (F1 vs F2)
    if tipo_fundacao == 'F2 (Mola Rotacional)':
        K_f = 300000 
        K_global[1, 1] += K_f  
        
        K_livre = K_global[1:6, 1:6]
        F_global = np.array([0.0, W_d[0], 0.0, W_d[1], 0.0])
        
        deslocamentos = np.linalg.solve(K_livre, F_global)
        u1, u2 = deslocamentos[1], deslocamentos[3]
        
    else: 
        K_livre = K_global[2:6, 2:6]
        F_global = np.array([W_d[0], 0.0, W_d[1], 0.0])
        deslocamentos = np.linalg.solve(K_livre, F_global)
        u1, u2 = deslocamentos[0], deslocamentos[2]

    # 5. Cálculos de Estabilidade (Gama-z)
    M1_tot_d = W_d[0] * H + W_d[1] * (2*H)
    Delta_M_tot_d = F_vd_pav * u1 + F_vd_pav * u2
    Gama_z = 1 / (1 - (Delta_M_tot_d / M1_tot_d))
    
    # 6. Saída Formatada
    print(f"--- RESULTADOS PARA A FUNDAÇÃO: {tipo_fundacao} ---")
    print(f"Deslocamento Nível 1: {u1*1000:.2f} mm")
    print(f"Deslocamento Nível 2: {u2*1000:.2f} mm")
    print(f"Momento Tombamento 1ª Ordem : {M1_tot_d:.2f} kNm")
    print(f"Momento 2ª Ordem (Delta M)  : {Delta_M_tot_d:.2f} kNm")
    print(f"Coeficiente Gama-z          : {Gama_z:.3f}\n")

# 7. Criando a Interface Interativa
dropdown_fundacao = widgets.Dropdown(
    options=['F1 (Engaste Perfeito)', 'F2 (Mola Rotacional)'],
    value='F1 (Engaste Perfeito)',
    description='Fundação:'
)

widgets.interact(analisar_estrutura_e2, tipo_fundacao=dropdown_fundacao);

interactive(children=(Dropdown(description='Fundação:', options=('F1 (Engaste Perfeito)', 'F2 (Mola Rotacional…

## 2. Distribuição de Sobrecarga e Momentos Excêntricos
Para a correta avaliação do multiplicador $\gamma$ de Hogeslag e da análise do pórtico, é necessário mapear o esforço normal máximo acumulado ($\Sigma F_{vd}$) e os momentos desequilibrados nos nós gerados pelas reações com excentricidade $e = 0,40~m$[cite: 2, 5].

O simulador abaixo avalia as tipologias **E1, E2 e E3**, permitindo o carregamento alternado da sobrecarga ($Q_k$) por pavimento para a determinação da envoltória crítica[cite: 2, 4].

In [2]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

def simulador_cargas_verticais(estrutura, carregar_pav1, carregar_pav2, carregar_pav3):
    num_pavimentos = int(estrutura[1]) 
    
    # Reações Isostáticas de Cálculo (kN) por vão de 18m (Ponderação 1.4)
    R_gk_calculo = 1.4 * (13 * 18 / 2)  
    R_qk_calculo = 1.4 * (7 * 18 / 2)   
    P_pilar_calculo = 1.4 * 24          
    
    e = 0.40  
    
    # Vetor de Presença da Sobrecarga [Pav1, Pav2, Pav3]
    presenca_qk = [carregar_pav1]
    if num_pavimentos >= 2: presenca_qk.append(carregar_pav2)
    if num_pavimentos >= 3: presenca_qk.append(carregar_pav3)
    
    N_externo = 0
    N_interno = 0
    
    for i, tem_sobrecarga in enumerate(presenca_qk):
        N_externo += P_pilar_calculo + R_gk_calculo + (R_qk_calculo if tem_sobrecarga else 0)
        N_interno += P_pilar_calculo + 2*R_gk_calculo + (2*R_qk_calculo if tem_sobrecarga else 0)
    
    F_vd_total = 2 * N_externo + 2 * N_interno 
    
    print(f"=== RESULTADOS PARA {estrutura.upper()} ===")
    print(f"Cenário de Sobrecarga: Nível 1: {presenca_qk[0]} " + 
          (f"| Nível 2: {presenca_qk[1]} " if num_pavimentos >= 2 else "") + 
          (f"| Nível 3: {presenca_qk[2]}" if num_pavimentos == 3 else ""))
    print("-" * 50)
    
    print("ESFORÇOS NORMAIS ACUMULADOS NA BASE (Pilar Isolado):")
    print(f"Pilar Externo : {N_externo:.2f} kN")
    print(f"Pilar Interno : {N_interno:.2f} kN")
    print(f"Somatório (ΣF_vd) do pórtico : {F_vd_total:.2f} kN\n")
    
    print("MOMENTOS EXCÊNTRICOS GERADOS NOS NÓS (Por pavimento):")
    for i, tem_sobrecarga in enumerate(presenca_qk):
        M_ext = (R_gk_calculo + (R_qk_calculo if tem_sobrecarga else 0)) * e
        M_int = (R_qk_calculo * e) if tem_sobrecarga else 0.0
        
        print(f"Pavimento {i+1} -> M_externo: {M_ext:.2f} kNm | M_interno (máx desequilíbrio): {M_int:.2f} kNm")

drop_estrutura = widgets.Dropdown(options=['E1', 'E2', 'E3'], value='E3', description='Estrutura:')
check_pav1 = widgets.Checkbox(value=True, description='Sobrecarga Nível 1 (Cota 6m)')
check_pav2 = widgets.Checkbox(value=True, description='Sobrecarga Nível 2 (Cota 12m)')
check_pav3 = widgets.Checkbox(value=True, description='Sobrecarga Nível 3 (Cota 18m)')

widgets.interact(simulador_cargas_verticais, 
                 estrutura=drop_estrutura, 
                 carregar_pav1=check_pav1, 
                 carregar_pav2=check_pav2, 
                 carregar_pav3=check_pav3);

interactive(children=(Dropdown(description='Estrutura:', index=2, options=('E1', 'E2', 'E3'), value='E3'), Che…